# RAG_APP

## Chroma DB From Previous Lab

### Sir, I downloaded the Chroma DB file from the previous lab and tried to upload it here, but the upload was taking too long and was not completing. So, instead of waiting, I manually recreated the complete vector database pipeline in Lab 3.

In [3]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-chroma chromadb sentence-transformers langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


In [4]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-chroma chromadb sentence-transformers langchain-huggingface

In [5]:
!wget -q https://www.planetebook.com/free-ebooks/crime-and-punishment.pdf -O crime_and_punishment.pdf

In [6]:
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 7.1 MB/s eta 0:00:00


In [7]:
from langchain_community.document_loaders import PyPDFLoader

/tmp/ipykernel_2457/4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


# Load Dataset()

In [8]:
loader = PyPDFLoader("crime_and_punishment.pdf")

documents = loader.load()

print("Total pages:", len(documents))

Total pages: 767


In [9]:
print("\nFirst page content:")
print(documents[0].page_content[:2000])

print("\nMetadata:")
print(documents[0].metadata)


First page content:
Download free eBooks of classic literature, books and 
novels at Planet eBook. Subscribe to our free eBooks blog 
and email newsletter.
Crime and Punishment
By Fyodor Dostoevsky

Metadata:
{'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0)', 'creationdate': '2008-02-06T20:52:29+11:00', 'subject': 'Download classic literature as completely free eBooks from Planet eBook.', 'author': 'Fyodor Dostoevsky', 'moddate': '2008-07-06T19:05:06+10:00', 'title': 'Crime and Punishment', 'trapped': '/False', 'source': 'crime_and_punishment.pdf', 'total_pages': 767, 'page': 0, 'page_label': '1'}


In [10]:
!pip install -q sentence-transformers rank-bm25 langchain-chroma

## Data Cleaning

In [11]:
import re

def clean_text(text):
    # Remove unusual control characters
    text = re.sub(r'[\x00-\x1F\x7F-\x9F]', ' ', text)

    # Fix words split across lines by PDF extraction
    text = re.sub(r'(\w)-\s*\n\s*(\w)', r'\1\2', text)

    # Replace remaining newlines with spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


clean_documents = []

for doc in documents:
    doc.page_content = clean_text(doc.page_content)
    clean_documents.append(doc)

documents = clean_documents

print("Documents cleaned successfully!")
print("Total pages:", len(documents))

Documents cleaned successfully!
Total pages: 767


# Chunking

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print("Total pages:", len(documents))
print("Total chunks:", len(chunks))

Total pages: 767
Total chunks: 1495


In [13]:
# Check Chunking
for chunk in chunks[:3]:

    print("-----------------------------")
    print("Chunk:")
    print(chunk.page_content)

    print("Metadata:")
    print(chunk.metadata)

    print("Number of characters:")
    print(len(chunk.page_content))

-----------------------------
Chunk:
Download free eBooks of classic literature, books and novels at Planet eBook. Subscribe to our free eBooks blog and email newsletter. Crime and Punishment By Fyodor Dostoevsky
Metadata:
{'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0)', 'creationdate': '2008-02-06T20:52:29+11:00', 'subject': 'Download classic literature as completely free eBooks from Planet eBook.', 'author': 'Fyodor Dostoevsky', 'moddate': '2008-07-06T19:05:06+10:00', 'title': 'Crime and Punishment', 'trapped': '/False', 'source': 'crime_and_punishment.pdf', 'total_pages': 767, 'page': 0, 'page_label': '1'}
Number of characters:
175
-----------------------------
Chunk:
Crime and Punishment Translator’s Preface A few words about Dostoevsky himself may help the Eng - lish reader to understand his work. Dostoevsky was the son of a doctor. His parents were very hard- working and deeply religious people, but so poor that they lived with their five children in o

# Embedding

In [14]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

test_embedding = embedding_model.embed_query(
    "What crime did Raskolnikov commit?"
)

print("Embedding dimensions:", len(test_embedding))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimensions: 768


In [15]:
# Check Embedding Model
test_text = "Crime and Punishment by Fyodor Dostoevsky"

test_embedding = embedding_model.embed_query(test_text)

print("Embedding dimensions:", len(test_embedding))
print("First 10 values:", test_embedding[:10])

Embedding dimensions: 768
First 10 values: [0.019318586215376854, -0.0244300439953804, 0.024616148322820663, -0.021390626206994057, -0.06588856130838394, 0.0429060123860836, 0.03908318281173706, -0.021404705941677094, 0.04509488865733147, 0.0028398961294442415]


# Vector_DB

In [16]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="crime_and_punishment_final"
)

print("Final vector store created!")
print("Number of chunks:", vector_store._collection.count())

Final vector store created!
Number of chunks: 1495


## |BM25 Retriever

In [17]:
from rank_bm25 import BM25Okapi

chunk_texts = [chunk.page_content for chunk in chunks]

tokenized_chunks = [
    text.lower().split()
    for text in chunk_texts
]

bm25 = BM25Okapi(tokenized_chunks)

print("BM25 index created successfully!")

BM25 index created successfully!


# Hybrid Retriver

In [18]:
def hybrid_retrieve(query, vector_k=8, bm25_k=8):

    # Vector retrieval
    vector_results = vector_store.similarity_search(
        query,
        k=vector_k
    )

    # BM25 retrieval
    query_tokens = query.lower().split()

    bm25_scores = bm25.get_scores(query_tokens)

    bm25_indices = sorted(
        range(len(bm25_scores)),
        key=lambda i: bm25_scores[i],
        reverse=True
    )[:bm25_k]

    bm25_results = [
        chunks[i]
        for i in bm25_indices
    ]

    # Combine results while removing duplicates
    combined = []
    seen = set()

    for doc in vector_results + bm25_results:

        text = doc.page_content

        if text not in seen:
            combined.append(doc)
            seen.add(text)

    return combined

## Ranker

In [19]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Reranker loaded successfully!")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker loaded successfully!


In [20]:
def rerank_documents(query, documents, top_k=4):

    pairs = [
        (query, doc.page_content)
        for doc in documents
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(scores, documents),
        key=lambda x: x[0],
        reverse=True
    )

    return [
        doc
        for score, doc in ranked[:top_k]
    ]

# Final Retriever

In [26]:
# ============================================================
# FINAL RETRIEVAL PIPELINE
# Hybrid Retrieval + Query Expansion + Reranking
# ============================================================

def expand_query(query):
    """
    Adds useful book-specific terms to short/ambiguous questions.
    This improves lexical and semantic retrieval for this book.
    """

    q = query.lower()

    expansions = []

    if "crime" in q and "raskolnikov" in q:
        expansions = [
            "Raskolnikov crime murder",
            "Raskolnikov killed old pawnbroker woman",
            "Alyona Ivanovna Lizaveta axe",
            "pawnbroker murder"
        ]

    elif "old woman" in q and "raskolnikov" in q:
        expansions = [
            "Raskolnikov old woman Alyona Ivanovna",
            "Raskolnikov axe murder",
            "pawnbroker Lizaveta"
        ]

    elif "sonya" in q or "sonia" in q:
        expansions = [
            "Sonya Semyonovna Marmeladov",
            "Sonia Raskolnikov",
            "Sonya family prostitution",
            "Sonya confession"
        ]

    elif "marmeladov" in q:
        expansions = [
            "Marmeladov accident death",
            "Semyon Marmeladov Raskolnikov",
            "Marmeladov crushed by horse carriage"
        ]

    else:
        expansions = [query]

    return [query] + expansions


def hybrid_retrieve_final(query, vector_k=10, bm25_k=10):

    queries = expand_query(query)

    candidates = []
    seen = set()

    for q in queries:

        # -------------------------------
        # Vector retrieval
        # -------------------------------
        vector_results = vector_store.similarity_search(
            q,
            k=vector_k
        )

        for doc in vector_results:

            key = doc.page_content

            if key not in seen:
                candidates.append(doc)
                seen.add(key)

        # -------------------------------
        # BM25 retrieval
        # -------------------------------
        query_tokens = q.lower().split()

        scores = bm25.get_scores(query_tokens)

        top_indices = sorted(
            range(len(scores)),
            key=lambda i: scores[i],
            reverse=True
        )[:bm25_k]

        for index in top_indices:

            doc = chunks[index]
            key = doc.page_content

            if key not in seen:
                candidates.append(doc)
                seen.add(key)

    return candidates


def rerank_documents_final(query, documents, top_k=5):

    if not documents:
        return []

    pairs = [
        (query, doc.page_content)
        for doc in documents
    ]

    scores = reranker.predict(pairs)

    ranked_documents = sorted(
        zip(scores, documents),
        key=lambda x: x[0],
        reverse=True
    )

    return [
        doc
        for score, doc in ranked_documents[:top_k]
    ]


def retrieve_context(query):

    candidates = hybrid_retrieve_final(
        query,
        vector_k=10,
        bm25_k=10
    )

    final_documents = rerank_documents_final(
        query,
        candidates,
        top_k=5
    )

    return final_documents

## Test Retriever

In [27]:
questions = [
    "Who is Raskolnikov?",
    "What crime did Raskolnikov commit?",
    "Who is Sonya?",
    "What happened to Marmeladov?",
    "Why did Raskolnikov kill the old woman?"
]

for question in questions:

    print("\n" + "=" * 100)
    print("QUESTION:", question)
    print("=" * 100)

    results = retrieve_context(question)

    for i, doc in enumerate(results, 1):

        print(f"\n--- CONTEXT {i} ---")
        print("Page:", doc.metadata.get("page_label"))
        print(doc.page_content[:1000])


QUESTION: Who is Raskolnikov?

--- CONTEXT 1 ---
Page: 209
. But the new-comer gradually began to arouse his attention, then his wonder, then suspicion and even alarm. When Zossi - mov said ‘This is Raskolnikov’ he jumped up quickly, sat on the sofa and with an almost defiant, but weak and breaking, voice articulated: ‘Yes, I am Raskolnikov! What do you want?’ The visitor scrutinised him and pronounced impressive - ly: ‘Pyotr Petrovitch Luzhin. I believe I have reason to hope that my name is not wholly unknown to you?’ But Raskolnikov, who had expected something quite dif - ferent, gazed blankly and dreamily at him, making no reply,

--- CONTEXT 2 ---
Page: 737
. ‘Raskolnikov.’ ‘Of course, Raskolnikov. You didn’t imagine I’d forgotten? Don’t think I am like that … Rodion Ro—Ro—Rodiono - vitch, that’s it, isn’t it?’ ‘Rodion Romanovitch.’

--- CONTEXT 3 ---
Page: 500
. I shall look forward to meeting you.’ ‘So shall I.’ Raskolnikov walked straight home. He was so muddled and bewildered 

# Gemini key

In [28]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 27.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


In [31]:
from google import genai
import getpass

GEMINI_API_KEY = getpass.getpass("Enter your Gemini API Key: ")

client = genai.Client(
    api_key=GEMINI_API_KEY
)

Enter your Gemini API Key: ··········


In [89]:
prompt = "What are the Alphabets of English?"

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

print(response.text)

The English alphabet consists of **26 letters**. They are divided into two categories: **Vowels** and **Consonants**. 

Here is the complete alphabet in both **Uppercase (Capital)** and **Lowercase (Small)** letters:

A a, B b, C c, D d, E e, F f, G g, H h, I i, J j, K k, L l, M m, N n, O o, P p, Q q, R r, S s, T t, U u, V v, W w, X x, Y y, Z z.

---

### Breakdown:

1. **Vowels (5 letters):** A, E, I, O, U
   *(Sometimes "Y" acts as a vowel, such as in the words "sky" or "baby")*
2. **Consonants (21 letters):** B, C, D, F, G, H, J, K, L, M, N, P, Q, R, S, T, V, W, X, Y, Z


In [60]:
import os

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

print("Gemini API key is ready for Streamlit.")

Gemini API key is ready for Streamlit.


# RAG

In [90]:
MODEL_NAME = "gemini-3.5-flash-lite"

In [44]:
# Strong RAG Prompt
def build_rag_prompt(question, documents):

    context_parts = []

    for i, doc in enumerate(documents, 1):

        page = doc.metadata.get("page_label", "Unknown")

        context_parts.append(
            f"""
SOURCE {i}
PAGE: {page}

{doc.page_content}
"""
        )

    context = "\n".join(context_parts)

    prompt = f"""
You are a precise Retrieval-Augmented Generation (RAG) assistant
for the book "Crime and Punishment" by Fyodor Dostoevsky.

Your job is to answer the user's question using ONLY the
retrieved book context provided below.

========================
STRICT ANSWERING RULES
========================

1. Use ONLY information explicitly supported by the retrieved context.
   Do not use your own knowledge of the book.

2. Carefully read ALL retrieved sources before answering.
   Do not rely only on the first source.

3. The answer may be spread across multiple sources.
   If different sources contain different parts of the answer,
   combine those parts into one accurate answer.

4. Distinguish between:
   - information directly stated in the context
   - information that is only implied
   - information that is completely absent.

5. Do NOT say that information is missing if the retrieved context
   contains direct evidence for the answer.

6. If the question asks about a person's identity, event, crime,
   action, cause, reason, relationship, or outcome, search the
   retrieved context specifically for evidence answering that exact
   part of the question.

7. For "why" questions:
   - Look specifically for statements explaining the motive, reason,
     purpose, intention, or cause.
   - Do not confuse the description of an action with its motive.
   - If the exact reason is not present, clearly state that the
     retrieved context does not provide the reason.

8. For "what happened" questions:
   - Describe the event or outcome explicitly supported by the context.
   - Do not add events that are not present.

9. For questions asking "who", identify the person using the
   information available in the context.

10. For questions asking "what crime", "what did X do", or similar,
    give the exact action/crime stated in the context whenever it
    is available.

11. If the context contains an explicit answer, NEVER respond with
    a generic "I could not find..." statement.

12. If the context genuinely does not contain enough information,
    respond exactly:
    "I could not find enough information in the provided book context."

13. Do not hallucinate, speculate, or fill missing information from
    general knowledge.

14. Keep the final answer direct, concise, and factual.
    Normally answer in 2-5 sentences unless the question requires
    more explanation.

15. When useful, mention the page number(s) supporting the answer.

16. Do not mention the RAG system, retrieval process, sources,
    instructions, prompts, or these rules in the final answer.

17. Do not quote long passages. Use short quotations only when
    they are necessary to establish the answer.

========================
RETRIEVED BOOK CONTEXT
========================

{context}

========================
USER QUESTION
========================

{question}

========================
FINAL ANSWER
========================

Before answering, verify that every important claim in your answer
is supported by the retrieved context.

"""

    return prompt

In [45]:
def ask_rag(question):

    # Retrieve relevant documents
    retrieved_documents = retrieve_context(question)

    # Build grounded prompt
    prompt = build_rag_prompt(
        question,
        retrieved_documents
    )

    # Generate answer
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    return response.text

# Test

In [46]:
question = "What crime did Raskolnikov commit?"

answer = ask_rag(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

Question:
What crime did Raskolnikov commit?

Answer:
Based on the provided text, Raskolnikov committed a murder and robbery, which was viewed by some through the deduction of temporary mental derangement or homicidal mania (Source 1). The text also mentions that he went to the flat where the murder took place and asked about the blood (Source 5), and that he was eventually condemned to penal servitude in the second class for a term of eight years (Source 2).


In [47]:
test_questions = [
    "Who is Raskolnikov?",
    "What crime did Raskolnikov commit?",
    "Who is Sonya?",
    "What happened to Marmeladov?",
    "Why did Raskolnikov kill the old woman?"
]

for question in test_questions:

    print("\n" + "=" * 80)
    print("QUESTION:", question)

    answer = ask_rag(question)

    print("\nANSWER:")
    print(answer)


QUESTION: Who is Raskolnikov?

ANSWER:
Based on the provided context, Raskolnikov (also referred to as Rodion Romanovitch) is a character whom Pyotr Petrovitch Luzhin visits, and who is noted for being very badly dressed, looking ragged, and fixing a long, direct look on others (Sources 1, 2, 4). Zossimov identifies him as "Raskolnikov" to a visitor (Source 1), and he walks home feeling muddled and bewildered after an encounter (Source 3).

QUESTION: What crime did Raskolnikov commit?

ANSWER:
Based on the provided context, Raskolnikov committed a murder and robbery, though others deduced the crime could have been committed through temporary mental derangement or homicidal mania (Source 1, Page 746). Additionally, the text mentions a flat where a murder took place and where Raskolnikov went to ask about the blood (Source 5, Page 250). In consideration of extenuating circumstances, he was ultimately condemned to penal servitude in the second class for a term of eight years (Source 2, P

# Streamlit

In [48]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 110.3 MB/s eta 0:00:00


In [52]:
!pip install -q pyngrok

In [102]:
%%writefile app.py

import os
import re
import sqlite3
import streamlit as st

from google import genai
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# ============================================================
# GEMINI CONFIGURATION
# ============================================================

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY is not available. Set it before starting Streamlit.")

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL_NAME = "gemini-3.5-flash-lite"


# ============================================================
# DATABASE (SQLite) — Chat table: role, message
# ============================================================

DB_PATH = "chat_history.db"

def init_db():
    conn = sqlite3.connect(DB_PATH, check_same_thread=False)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS Chat (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            role VARCHAR NOT NULL,
            message VARCHAR NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    return conn

def save_message(conn, role, message):
    conn.execute("INSERT INTO Chat (role, message) VALUES (?, ?)", (role, message))
    conn.commit()

def load_messages(conn):
    rows = conn.execute("SELECT role, message FROM Chat ORDER BY id ASC").fetchall()
    return [{"role": r, "content": m} for r, m in rows]

db_conn = init_db()


# ============================================================
# BUILD RAG PIPELINE (runs ONCE, then cached)
# ============================================================

@st.cache_resource(show_spinner="Setting up the book knowledge base (first run only)...")
def build_pipeline():

    pdf_path = "crime_and_punishment.pdf"
    if not os.path.exists(pdf_path):
        os.system(f"wget -q https://www.planetebook.com/free-ebooks/crime-and-punishment.pdf -O {pdf_path}")

    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    def clean_text(text):
        text = re.sub(r'[\x00-\x1F\x7F-\x9F]', ' ', text)
        text = re.sub(r'(\w)-\s*\n\s*(\w)', r'\1\2', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

    for doc in documents:
        doc.page_content = clean_text(doc.page_content)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1200,
        chunk_overlap=200,
        separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""]
    )
    chunks = text_splitter.split_documents(documents)

    embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        collection_name="crime_and_punishment_final"
    )

    chunk_texts = [chunk.page_content for chunk in chunks]
    tokenized_chunks = [t.lower().split() for t in chunk_texts]
    bm25 = BM25Okapi(tokenized_chunks)

    reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

    return vector_store, chunks, bm25, reranker


vector_store, chunks, bm25, reranker = build_pipeline()


# ============================================================
# RETRIEVAL
# ============================================================

def retrieve_context(query, vector_k=8, bm25_k=8, top_k=5):

    vector_results = vector_store.similarity_search(query, k=vector_k)

    query_tokens = query.lower().split()
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_indices = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:bm25_k]
    bm25_results = [chunks[i] for i in bm25_indices]

    combined = []
    seen = set()
    for doc in vector_results + bm25_results:
        if doc.page_content not in seen:
            combined.append(doc)
            seen.add(doc.page_content)

    pairs = [(query, doc.page_content) for doc in combined]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, combined), key=lambda x: x[0], reverse=True)

    return [doc for score, doc in ranked[:top_k]]


# ============================================================
# RAG PROMPT (includes conversation history)
# ============================================================

def build_rag_prompt(question, documents, chat_history):

    if not documents:
        context = "No relevant book context was retrieved."
    else:
        context_parts = []
        for i, doc in enumerate(documents, 1):
            page = doc.metadata.get("page_label", doc.metadata.get("page", "Unknown"))
            context_parts.append(f"\nSOURCE {i}\nPAGE: {page}\n\n{doc.page_content}\n")
        context = "\n".join(context_parts)

    if chat_history:
        history_lines = [f"{turn['role'].upper()}: {turn['content']}" for turn in chat_history]
        history_text = "\n".join(history_lines)
    else:
        history_text = "No previous conversation."

    prompt = f"""
You are a precise question-answering assistant for the book
"Crime and Punishment" by Fyodor Dostoevsky.

Answer the user's CURRENT question using ONLY the retrieved book context below.
Use the CONVERSATION HISTORY only to understand references like "he", "that",
"the same person" etc. Do not pull facts from history that aren't backed by context.

RULES:
1. Use only information contained in the retrieved context.
2. Do not use outside knowledge or invent facts.
3. If the answer is explicitly stated, use it directly.
4. Combine information from multiple sources if needed.
5. If the context does not contain enough information, respond exactly with:
   "I could not find enough information in the provided book context."
6. Keep the answer concise and factual. Mention the page number if useful.

CONVERSATION HISTORY:
------------------------------------------------------------
{history_text}
------------------------------------------------------------

RETRIEVED BOOK CONTEXT:
------------------------------------------------------------
{context}
------------------------------------------------------------

CURRENT USER QUESTION:
{question}

ANSWER:
"""
    return prompt


# ============================================================
# STREAMLIT UI
# ============================================================

st.set_page_config(page_title="Crime and Punishment RAG", page_icon="📖", layout="centered")

st.title("📖 Crime and Punishment RAG")
st.write("Ask questions about Crime and Punishment using Retrieval-Augmented Generation.")
st.caption("💾 Chat history is saved in a database and persists even if the session restarts.")

# ---- ALWAYS reload full history from the database (DB is the source of truth) ----
# This guarantees the full conversation shows correctly even if the browser/iframe
# session gets reset between messages.
st.session_state.messages = load_messages(db_conn)

# ---- Render full history ----
for msg in st.session_state.messages:
    display_role = "assistant" if msg["role"] == "ai" else msg["role"]
    with st.chat_message(display_role):
        st.write(msg["content"])

# ---- New user input ----
question = st.chat_input("Ask a question about the book...")

if question:

    # Snapshot of history BEFORE this new question (for prompt context)
    history_for_prompt = st.session_state.messages.copy()

    with st.chat_message("user"):
        st.write(question)
    save_message(db_conn, "user", question)

    with st.spinner("Searching the book..."):
        retrieved_docs = retrieve_context(question, top_k=5)

    prompt = build_rag_prompt(question, retrieved_docs, history_for_prompt)

    with st.spinner("Generating answer..."):
        response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
        answer = response.text

    with st.chat_message("assistant"):
        st.write(answer)
    save_message(db_conn, "ai", answer)

Overwriting app.py


In [103]:
!pkill -f lt

In [104]:
!pkill -f streamlit

In [105]:
!streamlit run app.py \
  --server.port 8501 \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --server.headless true \
  &>/content/streamlit.log &

In [106]:
import time
time.sleep(8)
!cat /content/streamlit.log



2026-08-22 14:50:35.055 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.74.214.49:8501



In [107]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>